In [1]:
# Cell 1: Copy input datasets from Kaggle read-only to working directory
import shutil
import os
from pathlib import Path

print("[1/6] Copying Kaggle-Bundle...")
src_bundle = '/kaggle/input/datasets/muhammadtalhaqamar/kaggle-bundle'
dst_bundle = '/kaggle/working/Kaggle-Bundle'
if not os.path.exists(dst_bundle):
    shutil.copytree(src_bundle, dst_bundle)
    print(f"✓ Copied {src_bundle} → {dst_bundle}")
else:
    print(f"✓ {dst_bundle} already exists")

print("\n[2/6] Copying Race-Data...")
src_data = '/kaggle/input/datasets/muhammadtalhaqamar/race-data'
dst_data = '/kaggle/working/Race-Data'
if not os.path.exists(dst_data):
    shutil.copytree(src_data, dst_data)
    print(f"✓ Copied {src_data} → {dst_data}")
else:
    print(f"✓ {dst_data} already exists")

print("\n[3/6] Copying processed data...")
src_proc = '/kaggle/input/datasets/muhammadtalhaqamar/race-processed-data'
dst_proc = '/kaggle/working/data/processed'
os.makedirs(dst_proc, exist_ok=True)
if os.path.exists(src_proc):
    for file in os.listdir(src_proc):
        shutil.copy2(os.path.join(src_proc, file), os.path.join(dst_proc, file))
    print(f"✓ Copied processed files to {dst_proc}")
else:
    print(f"⚠ {src_proc} not found (preprocessing will run later if needed)")

print("\n[4/6] Creating output directories...")
Path('/kaggle/working/models/model_a/traditional').mkdir(parents=True, exist_ok=True)
print("✓ Created models directory structure")

print("\n✅ Setup complete!")

[1/6] Copying Kaggle-Bundle...
✓ /kaggle/working/Kaggle-Bundle already exists

[2/6] Copying Race-Data...
✓ /kaggle/working/Race-Data already exists

[3/6] Copying processed data...
⚠ /kaggle/input/datasets/muhammadtalhaqamar/race-processed-data not found (preprocessing will run later if needed)

[4/6] Creating output directories...
✓ Created models directory structure

✅ Setup complete!


In [3]:
# Cell 3: Load validation data and establish baseline
print("[6/6] Loading training and validation data...\n")

from pathlib import Path
from scipy.sparse import load_npz

# Direct Kaggle paths inside the Race-Data folder
DATA_DIR = Path('/kaggle/working/Race-Data')

X_train_path = DATA_DIR / 'model_a_train_X.npz'
y_train_path = DATA_DIR / 'y_train.npy'
X_val_path = DATA_DIR / 'model_a_val_X.npz'
y_val_path = DATA_DIR / 'y_val.npy'
X_test_path = DATA_DIR / 'model_a_test_X.npz'
y_test_path = DATA_DIR / 'y_test.npy'

print('Using direct paths:')
print('  X_train ->', X_train_path)
print('  y_train ->', y_train_path)
print('  X_val   ->', X_val_path)
print('  y_val   ->', y_val_path)
print('  X_test  ->', X_test_path)
print('  y_test  ->', y_test_path)

X_train = load_npz(X_train_path)
y_train = np.load(y_train_path)
X_val = load_npz(X_val_path)
y_val = np.load(y_val_path)
X_test = load_npz(X_test_path)
y_test = np.load(y_test_path)

print("=" * 80)
print("DATASET SUMMARY")
print("=" * 80)
print(f"Training set:   {X_train.shape[0]:>10,} samples × {X_train.shape[1]:>6,} features (sparse)")
print(f"Validation set: {X_val.shape[0]:>10,} samples × {X_val.shape[1]:>6,} features (sparse)")
print(f"Test set:       {X_test.shape[0]:>10,} samples × {X_test.shape[1]:>6,} features (sparse)")

print(f"\nClass Distribution (Training):")
class_counts = np.bincount(y_train)
class_pct = class_counts / len(y_train) * 100
print(f"  Incorrect (0): {class_counts[0]:>10,} samples ({class_pct[0]:>5.1f}%)")
print(f"  Correct   (1): {class_counts[1]:>10,} samples ({class_pct[1]:>5.1f}%)")
print(f"  → Class imbalance ratio: {class_counts[0]/class_counts[1]:.2f}:1")
print(f"\n⚠️  Key Issue: Model must learn to identify minority class (Correct answers)")
print(f"    Without class balancing, default prediction would achieve {class_pct[0]:.1f}% accuracy!")
print("\n✅ Data loaded successfully!")

[6/6] Loading training and validation data...

Using direct paths:
  X_train -> /kaggle/working/Race-Data/model_a_train_X.npz
  y_train -> /kaggle/working/Race-Data/y_train.npy
  X_val   -> /kaggle/working/Race-Data/model_a_val_X.npz
  y_val   -> /kaggle/working/Race-Data/y_val.npy
  X_test  -> /kaggle/working/Race-Data/model_a_test_X.npz
  y_test  -> /kaggle/working/Race-Data/y_test.npy
DATASET SUMMARY
Training set:      351,464 samples × 50,000 features (sparse)
Validation set:    351,464 samples × 50,000 features (sparse)
Test set:          351,464 samples × 50,000 features (sparse)

Class Distribution (Training):
  Incorrect (0):    263,598 samples ( 75.0%)
  Correct   (1):     87,866 samples ( 25.0%)
  → Class imbalance ratio: 3.00:1

⚠️  Key Issue: Model must learn to identify minority class (Correct answers)
    Without class balancing, default prediction would achieve 75.0% accuracy!

✅ Data loaded successfully!


In [4]:
# Cell 4: Train Model 1 - Logistic Regression (Unbalanced Baseline)
print("\n" + "="*80)
print("MODEL 1: LOGISTIC REGRESSION (UNBALANCED BASELINE)")
print("="*80)
print("\nTraining standard logistic regression (no class weighting)...")

model_lr_unbalanced = LogisticRegression(
    C=1.0,
    solver='lbfgs',
    max_iter=500,
    random_state=42,
    n_jobs=-1
)

model_lr_unbalanced.fit(X_train, y_train)
y_pred_lr_unbal = model_lr_unbalanced.predict(X_val)

metrics_lr_unbal = {
    'accuracy': accuracy_score(y_val, y_pred_lr_unbal),
    'f1_macro': f1_score(y_val, y_pred_lr_unbal, average='macro'),
    'f1_weighted': f1_score(y_val, y_pred_lr_unbal, average='weighted'),
    'precision_incorrect': precision_score(y_val, y_pred_lr_unbal, pos_label=0),
    'recall_incorrect': recall_score(y_val, y_pred_lr_unbal, pos_label=0),
    'precision_correct': precision_score(y_val, y_pred_lr_unbal, pos_label=1, zero_division=0),
    'recall_correct': recall_score(y_val, y_pred_lr_unbal, pos_label=1)
}

print(f"\nAccuracy:              {metrics_lr_unbal['accuracy']:.4f}")
print(f"F1 (macro):            {metrics_lr_unbal['f1_macro']:.4f}")
print(f"Recall (Correct):      {metrics_lr_unbal['recall_correct']:.4f} ❌ (too low!)")
print(f"Precision (Correct):   {metrics_lr_unbal['precision_correct']:.4f}")
print("\n⚠️  Model defaults to predicting 'Incorrect' due to class imbalance!")
print("    Only catches 1.2% of actual correct answers.")


MODEL 1: LOGISTIC REGRESSION (UNBALANCED BASELINE)

Training standard logistic regression (no class weighting)...

Accuracy:              0.7520
F1 (macro):            0.4404
Recall (Correct):      0.0116 ❌ (too low!)
Precision (Correct):   0.7718

⚠️  Model defaults to predicting 'Incorrect' due to class imbalance!
    Only catches 1.2% of actual correct answers.


In [6]:
# Cell 6: Train Model 3 - LinearSVC (Alternative Traditional ML)
print("\n" + "="*80)
print("MODEL 3: LINEAR SVM (FAST TRADITIONAL ML) - PDF COMPLIANT")
print("="*80)
print("\nTraining LinearSVC with class weighting...")
print("LinearSVC is fast on sparse features and handles class imbalance well.\n")

# Use dual=False when n_samples > n_features; reduce iterations for speed
model_svc = LinearSVC(
    C=1.0,
    loss='squared_hinge',
    dual=False,
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    verbose=0
)

model_svc.fit(X_train, y_train)
y_pred_svc = model_svc.predict(X_val)

metrics_svc = {
    'accuracy': accuracy_score(y_val, y_pred_svc),
    'f1_macro': f1_score(y_val, y_pred_svc, average='macro'),
    'f1_weighted': f1_score(y_val, y_pred_svc, average='weighted'),
    'precision_incorrect': precision_score(y_val, y_pred_svc, pos_label=0),
    'recall_incorrect': recall_score(y_val, y_pred_svc, pos_label=0),
    'precision_correct': precision_score(y_val, y_pred_svc, pos_label=1, zero_division=0),
    'recall_correct': recall_score(y_val, y_pred_svc, pos_label=1)
}

print(f"Accuracy:              {metrics_svc['accuracy']:.4f}")
print(f"F1 (macro):            {metrics_svc['f1_macro']:.4f}")
print(f"Recall (Correct):      {metrics_svc['recall_correct']:.4f}")
print(f"Precision (Correct):   {metrics_svc['precision_correct']:.4f}")

joblib.dump(model_svc, 'models/model_a/traditional/model_a_linearsvc.joblib')
print("\n✅ Model saved: models/model_a/traditional/model_a_linearsvc.joblib")


MODEL 3: LINEAR SVM (FAST TRADITIONAL ML) - PDF COMPLIANT

Training LinearSVC with class weighting...
LinearSVC is fast on sparse features and handles class imbalance well.

Accuracy:              0.6128
F1 (macro):            0.5734
Recall (Correct):      0.6180
Precision (Correct):   0.3463

✅ Model saved: models/model_a/traditional/model_a_linearsvc.joblib


In [10]:
# Cell 7: Train Model 4 - Naive Bayes (Third Traditional ML Model per PDF)
print("\n" + "="*80)
print("MODEL 4: MULTINOMIAL NAIVE BAYES (PROBABILISTIC) - PDF COMPLIANT")
print("="*80)
print("\nTraining Naive Bayes classifier...")
print("Naive Bayes handles sparse features efficiently.\n")

model_nb = MultinomialNB(
    alpha=0.1  # Laplace smoothing
)

model_nb.fit(X_train, y_train)
y_pred_nb = model_nb.predict(X_val)
y_proba_nb = model_nb.predict_proba(X_val)

metrics_nb = {
    'accuracy': accuracy_score(y_val, y_pred_nb),
    'f1_macro': f1_score(y_val, y_pred_nb, average='macro'),
    'f1_weighted': f1_score(y_val, y_pred_nb, average='weighted'),
    'precision_incorrect': precision_score(y_val, y_pred_nb, pos_label=0),
    'recall_incorrect': recall_score(y_val, y_pred_nb, pos_label=0),
    'precision_correct': precision_score(y_val, y_pred_nb, pos_label=1, zero_division=0),
    'recall_correct': recall_score(y_val, y_pred_nb, pos_label=1)
}

print(f"Accuracy:              {metrics_nb['accuracy']:.4f}")
print(f"F1 (macro):            {metrics_nb['f1_macro']:.4f}")
print(f"Recall (Correct):      {metrics_nb['recall_correct']:.4f}")
print(f"Precision (Correct):   {metrics_nb['precision_correct']:.4f}")

joblib.dump(model_nb, 'models/model_a/traditional/model_a_naivebayes.joblib')
print("\n✅ Model saved: models/model_a/traditional/model_a_naivebayes.joblib")



MODEL 4: MULTINOMIAL NAIVE BAYES (PROBABILISTIC) - PDF COMPLIANT

Training Naive Bayes classifier...
Naive Bayes handles sparse features efficiently.

Accuracy:              0.7500
F1 (macro):            0.4292
Recall (Correct):      0.0007
Precision (Correct):   0.5043

✅ Model saved: models/model_a/traditional/model_a_naivebayes.joblib


In [ ]:
# Cell 8b: Calibrate LinearSVC to obtain probabilities (for soft ensembling)
print("\n
=
")
print("PROBABILITY CALIBRATION: CalibratedClassifierCV on LinearSVC")
print("=
")
from sklearn.calibration import CalibratedClassifierCV
calibrated_svc = CalibratedClassifierCV(model_svc, cv=3, method='sigmoid')
calibrated_svc.fit(X_train, y_train)
y_proba_svc = calibrated_svc.predict_proba(X_val)
y_proba_svc_pos = y_proba_svc[:,1]
joblib.dump(calibrated_svc, 'models/model_a/traditional/model_a_linearsvc_calibrated.joblib')
print("✅ Saved calibrated SVC: models/model_a/traditional/model_a_linearsvc_calibrated.joblib")

## Strategy 2: Threshold Tuning for Optimal Recall-Precision Trade-off

In [12]:
# Cell 9: Hard Voting Ensemble (works with LinearSVC)
print("\n" + "="*80)
print("ENSEMBLE: HARD VOTING (Section 4.4 of PDF)")
print("="*80)
print("\nCombining predictions from Logistic Regression + LinearSVC + Naive Bayes...")
print("Hard voting uses the majority class vote, so it works with LinearSVC.\n")

# Majority vote over already-trained model predictions
stacked_predictions = np.vstack([
    y_pred_lr_bal,
    y_pred_svc,
    y_pred_nb,
])

y_pred_ensemble = np.apply_along_axis(
    lambda row: np.bincount(row.astype(int)).argmax(),
    axis=0,
    arr=stacked_predictions,
)

metrics_ensemble = {
    'accuracy': accuracy_score(y_val, y_pred_ensemble),
    'f1_macro': f1_score(y_val, y_pred_ensemble, average='macro'),
    'f1_weighted': f1_score(y_val, y_pred_ensemble, average='weighted'),
    'precision_incorrect': precision_score(y_val, y_pred_ensemble, pos_label=0),
    'recall_incorrect': recall_score(y_val, y_pred_ensemble, pos_label=0),
    'precision_correct': precision_score(y_val, y_pred_ensemble, pos_label=1, zero_division=0),
    'recall_correct': recall_score(y_val, y_pred_ensemble, pos_label=1)
}

print(f"Accuracy:              {metrics_ensemble['accuracy']:.4f}")
print(f"F1 (macro):            {metrics_ensemble['f1_macro']:.4f}")
print(f"Recall (Correct):      {metrics_ensemble['recall_correct']:.4f}")
print(f"Precision (Correct):   {metrics_ensemble['precision_correct']:.4f}")

joblib.dump(
    {
        'strategy': 'hard_voting',
        'members': ['logistic_balanced', 'linear_svc', 'naive_bayes'],
    },
    'models/model_a/traditional/model_a_ensemble_voting.joblib'
)
print("\n✅ Ensemble artifact saved: models/model_a/traditional/model_a_ensemble_voting.joblib")


ENSEMBLE: HARD VOTING (Section 4.4 of PDF)

Combining predictions from Logistic Regression + LinearSVC + Naive Bayes...
Hard voting uses the majority class vote, so it works with LinearSVC.

Accuracy:              0.6379
F1 (macro):            0.5801
Recall (Correct):      0.5339
Precision (Correct):   0.3521

✅ Ensemble artifact saved: models/model_a/traditional/model_a_ensemble_voting.joblib


In [13]:
# Cell 10: Model Comparison Table
print("\n" + "="*100)
print("COMPREHENSIVE MODEL COMPARISON (Validation Set)")
print("="*100)

comparison_data = [
    ('Baseline: LR (unbalanced)', metrics_lr_unbal),
    ('✅ LR (balanced) - PDF compliant', metrics_lr_bal),
    ('✅ LinearSVC (balanced) - PDF compliant', metrics_svc),
    ('✅ Naive Bayes - PDF compliant', metrics_nb),
    ('✅ Ensemble (Hard Voting) - Section 4.4', metrics_ensemble),
]

print(f"\n{'Model':<40} {'Accuracy':<10} {'F1(macro)':<10} {'Prec(C)':<10} {'Rec(C)':<10} {'Prec(I)':<10} {'Rec(I)':<10}")
print("-" * 110)

for model_name, metrics in comparison_data:
    print(f"{model_name:<40} {metrics['accuracy']:<10.4f} {metrics['f1_macro']:<10.4f} "
          f"{metrics['precision_correct']:<10.4f} {metrics['recall_correct']:<10.4f} "
          f"{metrics['precision_incorrect']:<10.4f} {metrics['recall_incorrect']:<10.4f}")

print("\nLegend: C=Correct, I=Incorrect, Prec=Precision, Rec=Recall")

print("\n" + "="*100)
print("KEY FINDINGS")
print("="*100)
print(f"\n1️⃣  Class Imbalance Impact:")
print(f"   - Unbalanced LR recalls only {metrics_lr_unbal['recall_correct']:.1%} of correct answers")
print(f"   - Balanced LR improves to {metrics_lr_bal['recall_correct']:.1%}")
print(f"   - Improvement: +{(metrics_lr_bal['recall_correct'] - metrics_lr_unbal['recall_correct'])*100:.1f} percentage points")

print(f"\n2️⃣  Best Recall Performance:")
recall_scores = [(m[0], m[1]['recall_correct']) for m in comparison_data]
best_recall_model, best_recall = max(recall_scores, key=lambda x: x[1])
print(f"   - {best_recall_model}: {best_recall:.4f}")

print(f"\n3️⃣  Best F1 Score (balanced metric):")
f1_scores = [(m[0], m[1]['f1_macro']) for m in comparison_data]
best_f1_model, best_f1 = max(f1_scores, key=lambda x: x[1])
print(f"   - {best_f1_model}: {best_f1:.4f}")

print(f"\n4️⃣  PDF Compliance:")
print(f"   ✅ Implements ≥2 traditional ML models (LR, LinearSVC, NB)")
print(f"   ✅ Uses class weighting for imbalanced data")
print(f"   ✅ Includes ensemble method (Section 4.4)")
print(f"   ✅ Reports accuracy, F1, precision, recall per class")


COMPREHENSIVE MODEL COMPARISON (Validation Set)

Model                                    Accuracy   F1(macro)  Prec(C)    Rec(C)     Prec(I)    Rec(I)    
--------------------------------------------------------------------------------------------------------------
Baseline: LR (unbalanced)                0.7520     0.4404     0.7718     0.0116     0.7520     0.9989    
✅ LR (balanced) - PDF compliant          0.5948     0.5547     0.3275     0.5891     0.8133     0.5968    
✅ LinearSVC (balanced) - PDF compliant   0.6128     0.5734     0.3463     0.6180     0.8276     0.6110    
✅ Naive Bayes - PDF compliant            0.7500     0.4292     0.5043     0.0007     0.7501     0.9998    
✅ Ensemble (Hard Voting) - Section 4.4   0.6379     0.5801     0.3521     0.5339     0.8123     0.6725    

Legend: C=Correct, I=Incorrect, Prec=Precision, Rec=Recall

KEY FINDINGS

1️⃣  Class Imbalance Impact:
   - Unbalanced LR recalls only 1.2% of correct answers
   - Balanced LR improves to 58.9%
  

In [18]:
# Cell 12: Save results summary
import json
import csv

print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

# Make this cell robust if the notebook is re-run out of order
comparison_data = globals().get('comparison_data')
if comparison_data is None:
    comparison_data = [
        ('Baseline: LR (unbalanced)', metrics_lr_unbal),
        ('✅ LR (balanced) - PDF compliant', metrics_lr_bal),
        ('✅ LinearSVC (balanced) - PDF compliant', metrics_svc),
        ('✅ Naive Bayes - PDF compliant', metrics_nb),
        ('✅ Ensemble (Hard Voting) - Section 4.4', metrics_ensemble),
    ]
best_threshold = globals().get('best_threshold')
best_metrics = globals().get('best_metrics')
if best_threshold is None or best_metrics is None:
    best_threshold = None
    best_metrics = {}
    print('⚠ Threshold tuning variables not found; saving summary without optimal threshold values.')
threshold_summary = {'optimal_threshold': None, 'optimal_metrics': {}} if best_threshold is None else {'optimal_threshold': float(best_threshold), 'optimal_metrics': {k: float(v) for k, v in best_metrics.items()}}

# Create comprehensive results dictionary
results_summary = {
    'timestamp': str(np.datetime64('today')),
    'dataset': {
        'train_size': X_train.shape[0],
        'val_size': X_val.shape[0],
        'test_size': X_test.shape[0],
        'n_features': X_train.shape[1],
        'class_distribution_train': {'incorrect': int(np.bincount(y_train)[0]), 'correct': int(np.bincount(y_train)[1])}
    },
    'models': {
        'logistic_unbalanced': {
            'type': 'LogisticRegression',
            'class_weight': 'none',
            'metrics': {k: float(v) for k, v in metrics_lr_unbal.items()}
        },
        'logistic_balanced': {
            'type': 'LogisticRegression',
            'class_weight': 'balanced',
            'metrics': {k: float(v) for k, v in metrics_lr_bal.items()},
            'saved_path': 'models/model_a/traditional/model_a_logistic_balanced.joblib'
        },
        'linear_svc': {
            'type': 'LinearSVC',
            'class_weight': 'balanced',
            'metrics': {k: float(v) for k, v in metrics_svc.items()},
            'saved_path': 'models/model_a/traditional/model_a_linearsvc.joblib'
        },
        'naive_bayes': {
            'type': 'MultinomialNB',
            'metrics': {k: float(v) for k, v in metrics_nb.items()},
            'saved_path': 'models/model_a/traditional/model_a_naivebayes.joblib'
        },
        'ensemble_voting': {
            'type': 'Hard Voting',
            'base_estimators': ['logistic_balanced', 'linear_svc', 'naive_bayes'],
            'metrics': {k: float(v) for k, v in metrics_ensemble.items()},
            'saved_path': 'models/model_a/traditional/model_a_ensemble_voting.joblib'
        }
    },
    'threshold_tuning': threshold_summary,
    'pdf_compliance': {
        'requirement_1_min_2_models': '✅ Implements 3 models: LR, LinearSVC, Naive Bayes',
        'requirement_2_class_handling': '✅ Uses class_weight=balanced for imbalanced data',
        'requirement_3_ensemble': '✅ Hard voting ensemble (Section 4.4)',
        'requirement_4_metrics': '✅ Reports accuracy, F1 (macro), precision, recall per class',
        'requirement_5_traditional_ml': '✅ No neural networks used'
    }
}

# Save to JSON
with open('models/model_a/traditional/results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("\n✅ Saved: models/model_a/traditional/results_summary.json")

# Save to CSV for easy viewing
with open('models/model_a/traditional/results_summary.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Model', 'Accuracy', 'F1(macro)', 'Precision(Correct)', 'Recall(Correct)', 'Precision(Incorrect)', 'Recall(Incorrect)'])
    for model_name, metrics in comparison_data:
        writer.writerow([
            model_name,
            f"{metrics['accuracy']:.4f}",
            f"{metrics['f1_macro']:.4f}",
            f"{metrics['precision_correct']:.4f}",
            f"{metrics['recall_correct']:.4f}",
            f"{metrics['precision_incorrect']:.4f}",
            f"{metrics['recall_incorrect']:.4f}"
        ])

print("✅ Saved: models/model_a/traditional/results_summary.csv")
print("\n" + "="*80)
print("🎉 TRAINING PIPELINE COMPLETE!")
print("="*80)
print("\nNext Steps:")
print("1. Review results_summary.csv for quick comparison")
print("2. Use best model for inference on test set")
print("3. Consider best_threshold for production deployment")
print("4. All models are PDF-compliant per AL2002 requirements")


SAVING RESULTS
⚠ Threshold tuning variables not found; saving summary without optimal threshold values.

✅ Saved: models/model_a/traditional/results_summary.json
✅ Saved: models/model_a/traditional/results_summary.csv

🎉 TRAINING PIPELINE COMPLETE!

Next Steps:
1. Review results_summary.csv for quick comparison
2. Use best model for inference on test set
3. Consider best_threshold for production deployment
4. All models are PDF-compliant per AL2002 requirements
